In [1]:
!pip install -q transformers datasets huggingface_hub PyGithub peft bitsandbytes accelerate

## Step 1: Installing Modern ML Libraries
In this step, we install the core libraries required for our advanced code generation model:
* **`transformers` & `datasets`**: Hugging Face's foundational libraries for downloading models and managing data.
* **`PyGithub`**: To programmatically interact with GitHub and scrape high-quality code.
* **`peft` & `bitsandbytes` & `accelerate`**: The trifecta for modern, efficient fine-tuning. These allow us to use 4-bit Quantization (shrinking the model size) and LoRA (Low-Rank Adaptation) so we can train powerful models on a free Google Colab GPU without running out of memory.

In [4]:
from google.colab import userdata
from huggingface_hub import login

# Securely fetch tokens from Colab's Secrets manager using their NAMES
hf_token = userdata.get('HF_TOKEN')
github_token = userdata.get('GITHUB_TOKEN')

# Authenticate with Hugging Face Hub
login(token=hf_token)
print("Successfully authenticated with Hugging Face!")

Successfully authenticated with Hugging Face!


## Step 2: Secure Authentication
Security is a critical part of modern data science. Instead of hardcoding our private API tokens into the script (which could be accidentally leaked if we share the notebook), we use `google.colab.userdata` to pull them securely from Colab's encrypted Secrets manager. We then use the Hugging Face token to log in, granting us permission to push our final model to the Hub later.

In [5]:
from github import Github
import ast
from datasets import Dataset

# Initialize GitHub client
g = Github(github_token)

# Target a repository famous for high-quality algorithmic logic
repo = g.get_repo("TheAlgorithms/Python")

def extract_functions_ast(code):
    """Uses Python's Abstract Syntax Tree (AST) to extract valid functions."""
    functions = []
    try:
        # Parse the raw code into a structural tree
        tree = ast.parse(code)
        for node in ast.walk(tree):
            # Find strictly defined functions
            if isinstance(node, ast.FunctionDef):
                func_code = ast.get_source_segment(code, node)
                # Filter out uselessly short functions
                if func_code and len(func_code) > 100:
                    functions.append(func_code)
    except Exception:
        pass # Skip unparseable files
    return functions

print("Fetching files from GitHub...")
python_files = []
contents = repo.get_contents("")

# Limit to 75 files for a faster Colab training run
while contents and len(python_files) < 75:
    file_content = contents.pop(0)
    if file_content.type == "dir":
        contents.extend(repo.get_contents(file_content.path))
    elif file_content.path.endswith(".py"):
        python_files.append(file_content)

print("Extracting clean functions...")
data = {"code": []}
for file in python_files:
    try:
        code = file.decoded_content.decode("utf-8")
        data["code"].extend(extract_functions_ast(code))
    except Exception:
        continue

# Create the Hugging Face dataset and split it
dataset = Dataset.from_dict(data)
dataset = dataset.train_test_split(test_size=0.1)
print(f"Dataset ready! Training samples: {len(dataset['train'])}, Evaluation samples: {len(dataset['test'])}")

/tmp/ipython-input-427073679.py:6: DeprecationWarning: Argument login_or_token is deprecated, please use auth=github.Auth.Token(...) instead
  g = Github(github_token)


Fetching files from GitHub...
Extracting clean functions...
Dataset ready! Training samples: 144, Evaluation samples: 16


## Step 3: Creating a High-Quality Algorithmic Dataset
Garbage in, garbage out. To make our model smart, we are scraping `TheAlgorithms/Python`, a repository filled with classic computer science algorithms.

Instead of using messy Regular Expressions (Regex) to find code, we use Python's built-in `ast` module.  This parses the raw text into an Abstract Syntax Tree, allowing us to mathematically guarantee that we are only extracting perfectly formatted, complete Python functions. We also filter out any snippet under 100 characters to ensure the model learns complex logic.

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch

# We use a modern, highly capable coding model
model_id = "Qwen/Qwen2.5-Coder-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Configure 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading quantized model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Configure LoRA (Low-Rank Adaptation)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Tokenize the dataset
def preprocess(examples):
    return tokenizer(examples['code'], truncation=True, max_length=256, padding='max_length')

tokenized_datasets = dataset.map(preprocess, batched=True)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


Map:   0%|          | 0/144 [00:00<?, ? examples/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

## Step 4: Parameter-Efficient Fine-Tuning (PEFT) Setup
This is where the magic happens. We are using `Qwen2.5-Coder-0.5B`, a state-of-the-art small language model designed specifically for code.

To train this on a free GPU, we apply two massive optimizations:
1. **4-bit Quantization (`bitsandbytes`)**: We compress the model's weights into a smaller memory footprint without losing significant accuracy.
2. **LoRA (`peft`)**:  Instead of retraining all 500 million parameters of the model, LoRA freezes the original model and injects tiny, trainable "adapter" layers. Notice the printout above: we are only actually training about 1-2% of the total model parameters!

In [8]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# --- CHANGE THIS TO YOUR HUGGING FACE USERNAME ---
your_huggingface_username = "yashshah0211"
repo_name = f"{your_huggingface_username}/my-custom-algo-coder"

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch", # <-- This is the fixed line!
    fp16=True,
    push_to_hub=True,
    hub_model_id=repo_name
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("Starting training...")
trainer.train()

print("Pushing model to Hugging Face Hub...")
trainer.push_to_hub("Completed PEFT fine-tuning on Algorithms data")
print(f"Success! Model pushed to: https://huggingface.co/{repo_name}")

Starting training...


Epoch,Training Loss,Validation Loss
1,1.212768,1.165669
2,1.075039,1.142353
3,1.111491,1.136231


Pushing model to Hugging Face Hub...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...results/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...adapter_model.safetensors: 100%|##########| 4.34MB / 4.34MB            

  ...nt/results/tokenizer.json:  73%|#######2  | 8.30MB / 11.4MB            

Success! Model pushed to: https://huggingface.co/yashshah0211/my-custom-algo-coder


## Step 5: Training & Deployment
We set up our `TrainingArguments` to define how the model learns. We use `gradient_accumulation_steps` to simulate a larger batch size, which stabilizes training, and `fp16=True` to speed up the math using mixed precision.

Because we passed `push_to_hub=True`, the `Trainer` will automatically upload our fine-tuned LoRA weights directly to our Hugging Face profile once training is complete. You now have a hosted AI model in your portfolio!

In [9]:
def generate_advanced_code(prompt, max_length=150):
    # Prepare the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate using specific sampling settings
    outputs = model.generate(
        inputs['input_ids'],
        max_length=max_length,
        temperature=0.2,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test the newly trained model
prompt = "def binary_search(arr, target):"
print("🔍 Prompt:\n", prompt)
print("\n🧠 Generated Code:\n", generate_advanced_code(prompt))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔍 Prompt:
 def binary_search(arr, target):

🧠 Generated Code:
 def binary_search(arr, target):
    """
    Returns the index of the first occurrence of a target value in a sorted array
    using binary search.

    Args:
        arr (list): A sorted list of integers.
        target (int): The target value to search for.

    Returns:
        int: The index of the first occurrence of the target value, or -1 if the target is not found.
    """
    low = 0
    high = len(arr) - 1

    while low <= high:
        mid = (low + high) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1

    return -1


# Example usage:
if __name__ == "__main__":
    sorted_array = [1, 3, 5, 7, 9]
    target_value = 5
    index = binary_search(sorted_array, target_value)
    print(f"Index of {target_value} in the sorted array: {index}")


## Step 6: Testing with Advanced Inference Controls
Finally, we test our model. Unlike standard text generation where we want the AI to be highly creative, code needs to be logically sound.

We pass `temperature=0.2` and `top_p=0.9` to our generation function. A lower temperature forces the model to choose the most mathematically probable next tokens, making the resulting code much more predictable, deterministic, and less prone to syntax errors.